In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import json
import sys
import math
import matplotlib.pyplot as plt

In [2]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [3]:
from model import GPT, GPTConfig, GPTWithKVCache
config = GPTConfig()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", device)

In [4]:
from evaluation.metrics import (
    evaluate_perplexity,
    repetition_score,
    prefix_completion_test,
    nonsense_prompt_test,
    longest_token_match,
)
from training.data_utils import load_tokens

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT(config).to(device)
if device == "cpu":
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT, map_location="cpu")
else:
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT)

model.load_state_dict(ckpt)

model.eval()
print("Model loaded successfully")

Model loaded successfully


In [23]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    cfg.TOKENIZER_VOCAB,
    cfg.TOKENIZER_MERGES
)

In [7]:
train_tokens = np.memmap(
    cfg.TRAIN_TOKENS,
    dtype=np.uint16,
    mode="r"
)

val_tokens = np.memmap(
    cfg.VAL_TOKENS,
    dtype=np.uint16,
    mode="r"
)

test_tokens = np.memmap(
    cfg.TEST_TOKENS,
    dtype=np.uint16,
    mode="r"
)

print("Train tokens:", len(train_tokens))
print("Val tokens:", len(val_tokens))
print("Test tokens:", len(test_tokens))

Train tokens: 455046054
Val tokens: 2180159
Test tokens: 2395614


In [8]:
block_size = config.block_size

def get_batch(data, batch_size=16):

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([
        torch.from_numpy(data[i:i+block_size].astype(np.int64))
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64))
        for i in ix
    ])

    return x.to(device), y.to(device)

In [14]:
if device == "cpu":
  torch.set_num_threads(2)

In [17]:
%%time
num_batches = 100
val_loss = evaluate_dataset(val_tokens, num_batches)
print("Validation Loss:", val_loss)

test_loss = evaluate_dataset(test_tokens, num_batches)
print("Test Loss:", test_loss)

val_ppl = math.exp(val_loss)
test_ppl = math.exp(test_loss)

print("Validation Perplexity:", val_ppl)
print("Test Perplexity:", test_ppl)

# compute bits per byte
print("Validation Bits per Byte:", val_loss/math.log(2))
print("Test Bits per Byte:", test_loss/math.log(2))

Validation Loss: 1.8655173552036286
Test Loss: 1.8048584973812103
Validation Perplexity: 6.45927676215145
Test Perplexity: 6.0791111775049425
Validation Bits per Byte: 2.69137263704457
Test Bits per Byte: 2.6038604036781785
CPU times: user 25min 56s, sys: 4min 13s, total: 30min 10s
Wall time: 18min 8s


In [20]:
train_subset = train_tokens[:100000]

In [19]:
from generation.sampler import generate, encode_prompt


===== PREFIX COMPLETION TEST =====

PREFIX:
 his dad a big hug and walked home, carrying the turnip proudly in his hands.Once there

MODEL COMPLETION:
 his dad a big hug and walked home, carrying the turnip proudly in his hands.Once there was a little girl named Lisa. She loved to have fun. One day, her mummy took her out for a walk. It was a beautiful day and she was too excitement of it!  When they got home, mom asked Lisa to clean up her room for dinner. Lisa happily listened carefully as her mum began stacking the floor with soap and water. Yum! "I don't want to," Iâ€™ll like being wiped and cleaning in the kitchen so that she cleaned the room. Gradually she put the mess in the bathroom.  Her Mummy laughed and gave her a special

TRUE CONTINUATION FROM TRAIN DATA:
 his dad a big hug and walked home, carrying the turnip proudly in his hands.Once there was a mom and a dad who loved to share things with each other. They also loved to share things with their little girl. One day, the 


===== NONSENSE PROMPT TEST =====

PROMPT: Once upon a time there was a flying sandwich
Once upon a time there was a flying sandwichculars. She had many friends. Every day she would pack of her food and get ready for her own adventures. But when she around, she always forgot to pack.  One day when the flying birds fly around the flower around, the garden was especially very special. There were lots of wonderful plants and lots of flowers growing there. To make sure that the other bells were safe and green, the trees still packed up can of you!   From then on, Giraffe and scale visited each other's world. They were always so happy that each other was special to share the world.Lney wanted to find something exciting. She saw a big, soft structure in the grass with a blue ribbon. "What are these things?"

PROMPT: A dinosaur who loved pizza lived in a tiny house
A dinosaur who loved pizza lived in a tiny house. He had many favorite food, spaghetti, tomato sauce, and cheese. Every day he wo


===== REPETITION TEST =====
Repetition score: 0.06965174129353235
